# Параллелизм в Python

## Цели лекции

После этой лекции вы сможете:
- различать конкурентность и параллелизм, потоки и процессы;
- объяснять роль GIL и его влияние на CPU- и I/O-задачи;
- выбирать между `threading`, `multiprocessing` и `asyncio` под тип задачи;
- находить опасное чередование операций и корректно завершать фоновую работу.


### Модели конкурентности в Python

**Concurrency (конкурентность)** – это про то, как "заниматься множеством дел одновременно".  
**Parallelism (параллелизм)** – это про то, как "выполнять множество дел одновременно".  

> Эти понятия связаны, но не идентичны.  
> Одно касается структуры программы, другое – способа исполнения.

Иными словами:  
- **Конкурентность** – *модель организации кода*: способность обрабатывать несколько
  ожидающих задач, делая прогресс по одной за раз. Даже одноядерный процессор
  конкурентен, если ОС чередует выполнение задач.  
- **Параллелизм** – *способ выполнения*: несколько вычислений идут физически
  одновременно. Требует нескольких ядер, процессоров, GPU или узлов кластера.

Параллелизм – это частный случай конкурентности.  
Любая параллельная система конкурентна, но не любая конкурентная система – параллельна.


Модели конкурентности

### Общая картина

- Основная сложность конкурентного программирования – не запуск потоков или процессов,  
  а **управление** ими и **обмен данными**.  
- При обычном вызове `result = slow()` интерпретатор перейдёт к следующей строке  
  только после завершения `slow()`. Функция либо вернёт значение в `result`,  
  либо поднимет исключение прямо в месте вызова, где его можно перехватить.  
- Запуск потока или процесса возвращает управление сразу: фоновая работа ещё идёт.  
  Поэтому отдельно нужны ожидание завершения и канал для результата или ошибки.  
- Создание потоков и особенно процессов требует ресурсов. Для множества коротких
  заданий обычно используют пул долгоживущих воркеров.  
- У корутин место ожидания видно по `await`: через него же вызывающая корутина
  получает результат или исключение.


### Единицы выполнения

- **Единица выполнения (Execution unit)** – объект, исполняющий код независимо:  
  со своим состоянием и стеком вызовов.  
  В Python есть три таких типа: **процессы**, **потоки** и **корутины**.

- **Процесс (Process)** – выполняющаяся программа,  
  изолированная в своей памяти и получающая долю процессорного времени.  
  Общение между процессами происходит через **пайпы, сокеты, разделённую память** –  
  только в виде байтов. Для передачи объектов Python требуется **сериализация**,  
  что дорого по времени.

- **Поток (Thread)** – единица выполнения внутри одного процесса.  
  Все потоки процесса разделяют общую память.  
  Это упрощает обмен данными, но создаёт риск **гонки и логически неверного результата**,  
  если несколько потоков меняют одни данные. Риск есть и с GIL:  
  между проверкой и изменением данных может выполниться другой поток.  
  Потоки используют меньше ресурсов, чем процессы.

- **Корутина (Coroutine)** – функция, которая может приостанавливать и возобновлять работу.  
  Определяется через `async def`.  
  Обычно работает внутри одного потока под управлением **цикла событий (event loop)**.  
  Корутины реализуют **кооперативную многозадачность** –  
  управление передаётся явно через `await`.  
  Блокирующий код внутри корутины замораживает весь цикл событий.

### Очереди и блокировки

- **Очередь (Queue)** – структура данных для обмена элементами между потоками,  
  процессами или корутинами.  
  Чаще всего FIFO, но бывают и LIFO или приоритетные очереди.  
  Разные модели конкурентности используют свои реализации:  
  `queue.Queue`, `multiprocessing.Queue`, `asyncio.Queue`.

- **Блокировка (Lock / Mutex)** – объект синхронизации.  
  Позволяет ограничить одновременный доступ к разделяемым данным,  
  предотвращая их повреждение при обновлении.

### Процессы, потоки и знаменитая GIL

Далее рассматриваем **CPython с включённой GIL** – обычную сборку Python 3.13.
Есть и **free-threaded** сборка без GIL (например, `python3.14t`). Суффикс `t`
означает отдельную free-threaded сборку, а не ещё одну часть номера версии. В 3.13 она
экспериментальная, с 3.14 официально поддерживается, но команда `python`
по-прежнему запускает сборку с GIL. Проверить интерпретатор можно вызовом
`sys._is_gil_enabled()`. Защита общих данных нужна в обоих случаях.

1. Запуск программы командой `python` создаёт процесс с интерпретатором.
   Новые процессы можно запускать через `multiprocessing`, `concurrent.futures`
   или `subprocess` (для внешних программ).


2. Пользовательский код начинает выполняться в **главном потоке**.
   Дополнительные потоки создаются через `threading` или `concurrent.futures`.


3. **GIL (Global Interpreter Lock)** защищает внутреннее состояние интерпретатора.
   В одном интерпретаторе с включённой GIL только **один поток** за раз
   исполняет байткод Python. GIL не превращает несколько действий программы
   в одну неделимую операцию.


4. Потоки могут **по очереди** получать GIL.
   `sys.getswitchinterval()` возвращает целевой интервал переключения
   (обычно 0,005 с). Это **не гарантия переключения ровно каждые 5 мс**:
   влияют выполняемый код и планировщик ОС.


In [14]:
import sys
sys.getswitchinterval()

0.005

### Освобождение GIL и границы ускорения

5. Некоторые операции в C-расширениях **временно освобождают GIL**
   на время ожидания или вычислений, которым не нужен доступ к Python/C API.


6. Блокирующее ожидание ввода-вывода в стандартной библиотеке
   обычно освобождает GIL; например, ожидание сети или `time.sleep()`.
   Это не означает, что любой системный вызов обязательно освобождает GIL.


7. Некоторые численные операции NumPy и операции сжатия тоже
   освобождают GIL. Такие вычисления в C могут выполняться в потоках параллельно.
   Возможность ускорения зависит от конкретной операции и используемых данных.


8. Если задача большую часть времени ждёт сеть или диск,
   потоки позволяют перекрывать эти ожидания. Доля вычислений и длительность
   ожиданий определяют, насколько заметно влияние GIL.


9. В вычислениях на **чистом Python** с включённой GIL несколько
   потоков не исполняют байткод параллельно. Переключения и конкуренция
   за блокировку могут увеличить время работы.


10. Для таких вычислений на нескольких ядрах подходят **процессы**.
    Их запуск и обмен данными тоже стоят времени: ускорение нужно измерять,
    а не выводить только из количества воркеров.


### Как на практике запустить Python без GIL

У обычной сборки нельзя отключить GIL одним флагом. Сначала устанавливают
отдельную **free-threaded** сборку, затем запускают программу именно ею.
Если команда `python3.14t` не найдена, эта сборка ещё не установлена; обычный
`python3.14` её не заменяет. Команды ниже сработают только после установки:

```bash
# macOS / Linux: имя команды зависит от способа установки
python3.14t -X gil=0 -c "import sys; print(sys._is_gil_enabled())"

# Windows Python Launcher
py -3.14t -X gil=0 -c "import sys; print(sys._is_gil_enabled())"
```

Ожидаемый результат – `False`. Ключ `-X gil=1` временно включает GIL в той же
сборке. Если несовместимое C-расширение включит GIL, Python выведет предупреждение.

Сравнение одного и четырёх потоков на чистом Python:

```bash
python3.14 scripts/8_lesson/benchmark_threads.py
python3.14t scripts/8_lesson/benchmark_threads.py
```


### Гонка: один билет, два покупателя

Остался один билет. Каждый поток сначала проверяет наличие, затем оформляет покупку.
Между этими действиями может поработать другой поток – даже с включённой GIL.

| Шаг | Поток А | Поток Б |
|---|---|---|
| 1 | Видит один билет | |
| 2 | | Тоже видит один билет |
| 3 | Продаёт, записывает остаток 0 | |
| 4 | | Продаёт по прежнему решению, записывает 0 |

Остаток выглядит правдоподобно, но **продано два билета вместо одного**.
Защищать нужно правило «проверить и продать», а не только запись остатка.


### Воспроизводим ошибку с Barrier

- `Thread(target=buy_unsafe, args=(i,))` создаёт объект потока, но ещё не вызывает функцию.
- `start()` запускает поток: функция `buy_unsafe(i)` начинает выполняться,
  а главный поток сразу идёт дальше.
- `both_checked.wait()` останавливает каждого покупателя, пока оба потока не
  вызовут `wait()`. Когда приходит второй, барьер отпускает обоих. Они готовы
  продолжить, но какой поток выполнится первым, решает планировщик. Барьер нужен
  только для демонстрации: оба успевают вычислить `available` по старому значению.
- `join()` останавливает уже главный поток, пока покупатель не завершится.

Барьер делает неудачное чередование воспроизводимым; в рабочем коде продажу
защищает `Lock`, а не такой барьер.


In [1]:
from threading import Barrier, Thread

tickets = 1
sold = [False, False]
both_checked = Barrier(2, timeout=2)  # ждём двух покупателей

def buy_unsafe(index):
    global tickets
    available = tickets > 0           # каждый видит один билет
    both_checked.wait()                # оба уже проверили наличие
    if available:
        tickets = 0
        sold[index] = True

# Объекты потоков созданы, но buy_unsafe пока не запущена.
buyers = [Thread(target=buy_unsafe, args=(i,)) for i in range(2)]
for buyer in buyers:
    buyer.start()                     # запустить buy_unsafe(i)
for buyer in buyers:
    buyer.join(timeout=3)             # дождаться завершения потока
# Таймаут сам поток не останавливает, поэтому проверяем его состояние.
if any(buyer.is_alive() for buyer in buyers):
    raise TimeoutError("Покупатели ещё работают")
print("Продано:", sum(sold), "Осталось:", tickets)


Продано: 2 Осталось: 0


### Критическая секция охватывает всё решение

Оба покупателя используют **один и тот же Lock**. Проверка наличия и продажа
находятся внутри одного `with`. Если перенести проверку наружу, гонка останется.

Барьер больше не нужен. Нельзя переносить прежний `Barrier.wait()` внутрь Lock:
первый поток будет ждать второго, не пуская его к барьеру.


In [2]:
from threading import Lock

tickets = 1
sold = [False, False]
ticket_lock = Lock()

def buy_safe(index):
    global tickets
    with ticket_lock:
        if tickets > 0:
            tickets -= 1
            sold[index] = True

buyers = [Thread(target=buy_safe, args=(i,)) for i in range(2)]
for buyer in buyers:
    buyer.start()
for buyer in buyers:
    buyer.join(timeout=3)
if any(buyer.is_alive() for buyer in buyers):
    raise TimeoutError("Покупатели ещё работают")
print("Продано:", sum(sold), "Осталось:", tickets)


Продано: 1 Осталось: 0


### Конкурентный Hello World: спиннер на потоках

Цель примера: показать конкурентную работу двух задач в одном процессе с потоками.

- Пока «медленная» функция работает 3 секунды, в другом потоке крутится анимированный спиннер.
- Спиннер выводит символы из строки `"\|/-"` в одной позиции строки терминала.
- По завершении вычисления строка со спиннером очищается; печатается `Answer: 42`.
- Технические приёмы:
  - бесконечный цикл `itertools.cycle` для анимации;
  - возврат каретки `'\r'` для перерисовки одной строки;
  - сигнализация остановки через `threading.Event`.


In [2]:
# 1. spinner_thread.py: функции spin и slow
import itertools
import time
from threading import Thread, Event

def spin(msg: str, done: Event) -> None:
    for char in itertools.cycle(r'\|/-'):
        status = f'\r{char} {msg}'
        print(status, end='', flush=True)
        if done.wait(0.1):
            break
    blanks = ' ' * len(status)
    print(f'\r{blanks}\r', end='')

def slow() -> int:
    time.sleep(3)
    return 42


### Конкурентный Hello World: supervisor и main (потоки)

- `supervisor` координирует работу: запускает поток со спиннером, вызывает «медленную» функцию, останавливает спиннер.  
- Сигнализация остановки – через `threading.Event`: `done.set()` завершает цикл в `spin`.  
- Печать объекта потока полезна для наглядности: видно состояние (`initial` до `start`).  
- Остановка и ожидание помещены в `finally`, чтобы выполнить их и при ошибке `slow()`.
- `main` вызывает `supervisor` и печатает результат.  
- Предполагается, что `spin` и `slow` определены выше.


In [3]:
# 2. spinner_thread.py: supervisor и main
from threading import Thread, Event

def supervisor() -> int:
    done = Event()
    spinner = Thread(target=spin, args=('thinking!', done))
    print(f'spinner object: {spinner}')
    spinner.start()
    try:
        return slow()
    finally:
        done.set()       # spin проверяет событие и выходит из цикла
        spinner.join()   # ждём завершения перед выходом из supervisor

def main() -> None:
    result = supervisor()
    print(f'Answer: {result}')

if __name__ == '__main__':
    main()


spinner object: <Thread(Thread-5 (spin), initial)>
Answer: 42  


### Ошибка основной работы не должна оставлять спиннер

Проследим уже написанный `supervisor`, если `slow()` поднимет `ValueError`:

1. Выполнение `try` прерывается.
2. `finally` отправляет `done.set()` и ждёт завершения спиннера.
3. Исходный `ValueError` выходит из `supervisor` к вызывающему коду.

При обычном результате `finally` тоже выполнится **до возврата** из функции.
Если поставить `done.set()` и `spinner.join()` после `slow()`, но **вне `finally`**,
исключение перепрыгнет через эти строки: спиннер не получит сигнал и продолжит работать.

Это применение разобранного ранее `finally` к времени жизни фоновой работы.
Ожидание завершится, потому что наш `spin` регулярно проверяет событие.


### Исключение внутри потока не выходит из join

В спиннере ошибка возникала в основной работе. Теперь падает сам воркер:

```python
from threading import Thread

def fail():
    raise ValueError("Некорректное задание")

worker = Thread(target=fail)
worker.start()
worker.join()
print("Основной поток продолжает работу")
```

В консоли будет трассировка ошибки, но основной поток напечатает свою строку.
Обычный `try/except ValueError` вокруг `join()` не поймает ошибку из `fail`.
Необработанное исключение попадает в `threading.excepthook`; по умолчанию он
печатает его в stderr. Порядок сообщений в stdout и stderr может различаться.


### Результат или ошибка: явный ответ воркера

Воркер отправляет ровно одно сообщение через `replies.put(...)`, а основной поток
получает его вызовом `replies.get(...)`. Когда `get()` вернул кортеж, воркер уже
выполнил `put()`, но сам поток может ещё завершаться. Присваивание только распаковывает
кортеж в `status` и `value`.

`timeout=2` означает одно ожидание длительностью не более двух секунд, а не две
попытки. Если ответа нет, `get()` поднимает `queue.Empty`. `join()` после этого
отдельно ждёт завершения потока.

Сообщение содержит статус и значение либо описание ошибки. Основной поток решает,
что делать с этим ответом; сообщение об ошибке не означает успешное выполнение задания.


In [26]:
from queue import Queue
from threading import Thread

def parse_count(text, replies):
    try:
        value = int(text)
    except ValueError as error:
        replies.put(("error", f"{type(error).__name__}: {error}"))
    else:
        replies.put(("ok", value))

for text in ("12", "двенадцать"):
    replies = Queue()
    worker = Thread(target=parse_count, args=(text, replies))
    worker.start()
    try:
        status, value = replies.get(timeout=2)
    finally:
        worker.join(timeout=2)
    if worker.is_alive():
        raise TimeoutError("Воркер ещё работает")
    print(status, value)


ok 12
error ValueError: invalid literal for int() with base 10: 'двенадцать'


### Поток ждёт события, join ждёт поток

`Thread(target=stop.wait)` запускает в новом потоке метод `stop.wait`.
Пока флаг события сброшен, `wait()` не возвращает управление и поток остаётся жив.
`stop.set()` устанавливает флаг, `wait()` возвращается, целевая функция заканчивается –
после этого завершается и поток.

`join()` без таймаута приостанавливает вызывающий поток, пока целевой не завершится.
`join(timeout=...)` ждёт не дольше указанного времени и возвращается раньше, если
целевой поток успел завершиться. Если таймаут истёк, целевой поток продолжает работу.
Оба варианта возвращают `None`, поэтому после таймаута проверяют `is_alive()`.


In [3]:
from threading import Event, Thread

stop = Event()
waiting = Thread(target=stop.wait)
waiting.start()
try:
    result = waiting.join(timeout=0.01)
    print("Результат join:", result)
    print("До сигнала:", waiting.is_alive())
finally:
    stop.set()
    waiting.join(timeout=2)
print("После сигнала и ожидания:", waiting.is_alive())


Результат join: None
До сигнала: True
После сигнала и ожидания: False


### Event не прерывает произвольную работу

`stop.set()` сразу устанавливает флаг, но не завершает поток принудительно.
Код, который должен остановиться, сам вызывает `wait()` или проверяет `is_set()`.
В нашем примере это дочерний поток. `wait(timeout)` возвращает `True`, если пришёл
сигнал, и `False`, если истёк таймаут: тогда цикл может выполнить следующий шаг.
Без таймаута `wait()` будет спать до `set()` и не сможет периодически обновлять спиннер.

| Чем занят поток | Что даёт `stop.set()` |
|---|---|
| Ждёт `stop.wait(10)` | Ожидание проснётся сразу, не нужно ждать все 10 секунд |
| Выполняет `time.sleep(10)` | Сон продолжится; событие его не прерывает |
| Ждёт блокирующий сетевой вызов | Нужен таймаут самого вызова; затем можно проверить событие |

Если после сигнала и повторного `join(timeout)` поток ещё жив,
его работу нельзя считать завершённой. Разбейте долгую работу на шаги
с проверками события и ограничьте внешние ожидания.


### Взаимная блокировка: два потока ждут друг друга

**Deadlock** возникает, если продолжение каждого зависит от другого.
Например, покупка и возврат используют блокировки мест и кассы в разном порядке:

| Шаг | Покупка | Возврат |
|---|---|---|
| 1 | Захватила `seat_lock` | Захватил `cash_lock` |
| 2 | Ждёт `cash_lock` | Ждёт `seat_lock` |
| 3 | Не может освободить место до продолжения | Не может освободить кассу до продолжения |

Получился цикл ожидания. Ни GIL, ни дополнительные ядра его не разорвут.
Таймаут у `join` вернёт управление ожидающему коду, но не освободит эти блокировки.


### Единый порядок захвата разрывает цикл

Договоримся: **сначала места, затем касса** во всех операциях, включая возврат.
Второй поток будет ждать первую блокировку, ещё не удерживая вторую.
Пример показывает только порядок захвата двух общих ресурсов.


In [5]:
from threading import Lock, Thread

seat_lock = Lock()
cash_lock = Lock()
completed = [False, False]

def update_booking(index):
    with seat_lock:
        with cash_lock:
            completed[index] = True

operations = [Thread(target=update_booking, args=(i,)) for i in range(2)]
for operation in operations:
    operation.start()
for operation in operations:
    operation.join(timeout=2)
if any(operation.is_alive() for operation in operations):
    raise TimeoutError("Операции ещё работают")
print("Завершены обе операции:", all(completed))


Завершены обе операции: True


### Спиннер на процессах (multiprocessing)

- `multiprocessing` запускает **отдельный интерпретатор** Python как дочерний процесс.  
- У каждого процесса свой **GIL** → можно задействовать **все ядра CPU** (решает ОС-планировщик).  
- API намеренно похож на `threading`: `Process`, `Event`, `start()`, `join()`.  
- При переходе с потоков на процессы важны **коммуникации** и **сериализация** данных.  
- В этом примере через границу процессов проходит только состояние `Event` (на ОС-семафоре).  
- Сигнализация остановки – так же через `Event.set()`.  
- Типизации: `multiprocessing.Event` – **функция**, возвращает `synchronize.Event` (для подсказок типов).

Код ниже показан как содержимое `.py`-файла. Особенности запуска из Jupyter
и требования `spawn` разберём на следующих подслайдах.


In [ ]:
# 1. spinner_proc.py – изменившиеся части (остальное как в spinner_thread.py)
import itertools
import time
from multiprocessing import Process, Event
from multiprocessing import synchronize  # для корректной аннотации типов Event

def spin(msg: str, done: synchronize.Event) -> None:
    # остальной код идентичен версии на потоках:
    for char in itertools.cycle(r'\|/-'):
        status = f'\r{char} {msg}'
        print(status, end='', flush=True)
        if done.wait(0.1):
            break
    blanks = ' ' * len(status)
    print(f'\r{blanks}\r', end='')

def slow() -> int:
    time.sleep(3)
    return 42

def supervisor() -> int:
    done = Event()  # возвращает synchronize.Event
    spinner = Process(target=spin, args=('thinking!', done))
    print(f'spinner object: {spinner}')  
    spinner.start()
    try:
        return slow()
    finally:
        done.set()
        spinner.join()

# без изменений
def main() -> None:
    print(f'Answer: {supervisor()}')

if __name__ == '__main__':
    main()


In [6]:
import subprocess
import sys
from pathlib import Path

script = Path('../scripts/8_lesson/spinner_proc.py')
if not script.exists():
    script = Path('scripts/8_lesson/spinner_proc.py')

completed = subprocess.run(
    [sys.executable, script],
    capture_output=True, text=True, check=True,
)
parts = completed.stdout.replace('\r', '\n').splitlines()
print('Спиннер работал:', any(part.strip().endswith('thinking!') for part in parts))
print(next(part for part in reversed(parts) if part.startswith('Answer:')))


Спиннер работал: True
Answer: 42


### spawn: дочерний процесс импортирует код заново

При `spawn` запускается свежий интерпретатор. Он загружает главный модуль,
чтобы найти функцию, указанную в `target`.

`родитель → новый Python → загрузка модуля → вызов target(*args)`

Родитель запускает `main.py` как главный модуль. Новый интерпретатор загружает
тот же файл, чтобы получить функцию `target`. Если `Process(...).start()` стоит
на верхнем уровне, дочерний процесс при загрузке снова дойдёт до этой строки и
попытается создать следующий процесс. Поэтому запуск помещаем под знакомое условие
`if __name__ == "__main__"`: в родителе оно истинно, а при загрузке дочерним
интерпретатором – ложно.

Саму рабочую функцию оставляем на верхнем уровне файла: дочерний интерпретатор
должен найти её по имени после загрузки модуля.

При `spawn` и `forkserver` главный модуль должен допускать безопасную повторную загрузку.


### Запускаемый пример spawn из файла

`spawn` – один из **методов старта** процесса. Прямой API `multiprocessing`
использует контекст по умолчанию либо выбранный через `set_start_method()`.
`get_context("spawn")` ничего не запускает: он локально возвращает тот же API,
привязанный к `spawn`, и не меняет глобальную настройку программы. Поэтому
пример одинаково работает на Windows, macOS и Linux.

Файл [`scripts/8_lesson/spawn_demo.py`](../scripts/8_lesson/spawn_demo.py):

```python
from multiprocessing import get_context


def show_square(number):
    print("Квадрат:", number * number, flush=True)


def main():
    context = get_context("spawn")
    process = context.Process(target=show_square, args=(7,))
    process.start()
    try:
        process.join(timeout=5)
        if process.is_alive():
            raise TimeoutError("Дочерний процесс не завершился")
        if process.exitcode != 0:
            raise RuntimeError(f"Код завершения: {process.exitcode}")
    finally:
        if process.is_alive():
            process.terminate()
        process.join()
        process.close()


if __name__ == "__main__":
    main()
```

Из корня репозитория: `python scripts/8_lesson/spawn_demo.py` → `Квадрат: 49`.
В файле есть аварийное завершение при таймауте: здесь процесс только считает число.
Для воркеров с очередями и блокировками принудительное завершение может повредить обмен.


### Что можно передать дочернему процессу

Для `spawn` функция должна быть доступна по имени из загружаемого модуля,
а аргументы – допускать межпроцессную передачу (обычно через `pickle`).

| Передаём | Результат |
|---|---|
| Функцию верхнего уровня `.py` и число `7` | Подходит для нашего примера |
| `lambda` или вложенную функцию | Обычный `pickle` не сможет передать такую функцию |
| `threading.Lock()` | Блокировка потока не передаётся; нужен примитив из контекста `multiprocessing` |

Функция из ячейки Jupyter обычно недоступна новому интерпретатору по имени.
Одного условия `__main__` в ячейке недостаточно: перенесите пример в `.py`
и запустите из терминала. Признаки проблемы: ошибка сериализации при `start()`
или ошибка поиска функции уже в дочернем процессе.


### Спиннер на корутинах (asyncio)

- **Цикл событий** управляет очередью задач и переключается между ними, когда текущая задача ждёт.
- В примере используется:
  - `asyncio.run()` – запускает цикл событий и выполняет корутину до конца;
  - `asyncio.create_task()` – создаёт задачу (`Task`), которая выполняется конкурентно с текущей корутиной;
  - `await` – приостанавливает текущую корутину, пока не будет готов ожидаемый объект;
  - `spinner.done()` – проверяет, завершилась ли задача результатом, ошибкой или отменой.
    Сразу после `create_task()` здесь получаем `False`.
- Остановка спиннера: `spinner.cancel()` **запрашивает** отмену – внутри `spin` поднимется
  `CancelledError`; `await spinner` ждёт, пока задача завершится.

В `.py`-файле корутину запускает `asyncio.run()`. В Jupyter цикл событий уже работает,
поэтому после определений вызываем `await supervisor()` прямо в ячейке.


In [9]:
# 1. spinner_async.py: main и supervisor
import asyncio

async def supervisor() -> int:
    spinner = asyncio.create_task(spin('thinking!'))
    print(f'spinner task: done={spinner.done()}')
    try:
        return await slow()            # ожидаем завершения "медленной" корутины
    finally:
        spinner.cancel()               # запрашиваем отмену спиннера
        try:
            await spinner              # ждём, пока spin очистит строку и завершится
        except asyncio.CancelledError:
            pass                       # отмену запросил сам supervisor

def main() -> None:
    result = asyncio.run(supervisor())  # запускаем event loop и корутину supervisor
    print(f'Answer: {result}')

# В файле spinner_async.py:
# if __name__ == '__main__':
#     main()


### Спиннер на корутинах: функции `spin` и `slow`

- Паузы – через `await asyncio.sleep(...)`: на время ожидания управление уходит циклу событий.
- После `cancel()` исключение `CancelledError` поднимется в `spin` на ближайшем `await`.
- `spin` его не перехватывает: очистка строки стоит в `finally` и выполнится при отмене.
- Если перехватить `CancelledError` и не поднять снова, отмена потеряется: задача завершится
  как обычно, и вызывающий код не узнает, что её прервали.
- `supervisor` сам запросил отмену, поэтому при ожидании `spinner` перехватывает `CancelledError`:
  для него это ожидаемый исход.
- `slow` – асинхронная «медленная» функция: `await asyncio.sleep(3)` → `42`.


In [10]:
# 2. spinner_async.py: корутины spin и slow
import asyncio
import itertools

async def spin(msg: str) -> None:
    status = ''
    try:
        for char in itertools.cycle(r'\|/-'):
            status = f'\r{char} {msg}'
            print(status, flush=True, end='')
            # неблокирующая пауза: даём event loop переключиться на другие корутины
            await asyncio.sleep(0.1)
    finally:
        # выполняется и при отмене задачи: очищаем строку спиннера
        blanks = ' ' * len(status)
        print(f'\r{blanks}\r', end='')

async def slow() -> int:
    # имитация длительной операции без блокировки event loop
    await asyncio.sleep(3)
    return 42

result = await supervisor()
print(f'Answer: {result}')

spinner task: done=False
Answer: 42


### «Сломаем» спиннер для понимания asyncio

Рассмотрим, почему нельзя вызывать `time.sleep(...)` внутри корутин.

Заменим в `slow()` строку `await asyncio.sleep(3)` на `time.sleep(3)`.

Ожидаемо:
- Печатается состояние задачи спиннера: `done=False`.
- Анимация не появляется; программа «застывает» на 3 секунды.
- Затем выводится `Answer: 42`, и программа завершается.

Вывод:
- `time.sleep` блокирует **весь** поток и **event loop**.
- Для пауз в корутинах используйте только `await asyncio.sleep(...)`.


In [6]:
# spinner_async_experiment.py: отличается от spinner_async.py только функцией slow
import time

async def slow() -> int:
    # Неправильно: блокирует весь event loop на 3 секунды
    time.sleep(3)
    return 42

result = await supervisor()
print(f'Answer: {result}')


spinner task: done=False
Answer: 42


### Супервизоры: сравнение подходов

Ключевые различия и сходства:

- `Task` (asyncio) ≈ `Thread` (threading), но:
  - `Task` запускается через `asyncio.create_task(coro())` и **сразу планируется**.
  - `Thread` нужно **явно** запускать `start()`.
- Что исполняют:
  - `Task` **ведёт корутину**; переключение – через `await`.
  - `Thread` **вызовет функцию**.
- Ожидание «медленной» работы:
  - В потоках `slow()` – **обычная функция**, вызывается напрямую.
  - В asyncio `slow()` – **корутина**, ожидается через `await slow()`.
- Завершение «спиннера» – запрос и ожидание:
  - Потоки: **нет API убийства** потока → сигнал `Event.set()`, затем `join()`.
  - asyncio: `task.cancel()` → внутри корутины поднимается `CancelledError`, затем `await task`.
- Запуск всей конструкции:
  - Потоки: достаточно вызвать `supervisor()` из `main`.
  - asyncio: корутина-супервизор запускается через `asyncio.run(...)`.

Вывод: в потоках синхронизация достигается примитивами (Event/Lock), в asyncio – **структурой `await`** и управлением через `Task`.


### Реальное влияние GIL

- В потоковой версии ожидание сети, как и `time.sleep(3)`, обычно
  освобождает GIL: спиннер продолжает вращаться.
- В асинхронной версии сетевое ожидание должно отдавать управление
  циклу событий, как `await asyncio.sleep(3)`.
- **Вычисления на чистом Python не останавливают все остальные потоки**:
  интерпретатор чередует их выполнение, но с включённой GIL они не считают
  одновременно на разных ядрах.
- **C-операция** здесь – работа внутри интерпретатора или модуля, написанного
  на C, а не цикл байткода Python. Автор C-кода может временно освободить GIL,
  пока не обращается к объектам Python. Если длительная операция этого не делает,
  другие потоки с Python-кодом ждут её завершения.

Сравним три спиннера на Python-функции проверки простого числа.


In [25]:
import math

def is_prime(n: int) -> bool:
    if n < 2:
        return False
    if n == 2:
        return True
    if n % 2 == 0:
        return False
    root = math.isqrt(n)
    for i in range(3, root + 1, 2):
        if n % i == 0:
            return False
    return True

print(is_prime(5_000_111_000_222_021))  

True


### Мини-викторина: влияние GIL на спиннер

**Вопрос:**  
Что произойдёт со спиннером, если заменить `sleep(3)` на `is_prime(n)` (где `n = 5_000_111_000_222_021`)  
в трёх версиях программы?

1. В `spinner_proc.py`  
2. В `spinner_thread.py`  
3. В `spinner_async.py`

Ответ проверим запуском из корня репозитория: `python scripts/8_lesson/spinner_thread.py prime`.
Так же запускаются `spinner_proc.py` и `spinner_async.py`; без `prime` скрипты ждут `sleep(3)`.


### Ответы: как ведёт себя спиннер

**multiprocessing**

- Спиннер работает в **дочернем процессе**, а `is_prime(n)` – в **родительском**.  
- У каждого процесса свой интерпретатор и **свой GIL**.  
- При наличии свободных ядер ОС может выполнять их **параллельно**.  
- Спиннер **продолжает вращаться**, пока основной процесс считает простое число.


**threading**

- Спиннер работает во **вторичном потоке**, `is_prime(n)` – в **главном**.
- GIL периодически переходит между потоками: спиннер получает возможность
  напечатать кадр. Он продолжает вращаться, хотя основной поток занят CPU.
- Два потока чередуются, но не исполняют Python-байткод одновременно.
- Плавность спиннера и накладные расходы зависят от нагрузки и планирования.
  Добавление вычисляющих потоков не гарантирует ускорения; оно может замедлить программу.


**asyncio**

- `slow` и спиннер используют один поток цикла событий.
- Вызов `is_prime(n)` внутри `slow` не отдаёт управление циклу событий:
  спиннер не получает возможности сделать очередной шаг.
- Он может даже не успеть показать первый кадр перед отменой.
- Причина – **отсутствие уступки управления**, а не GIL.
  Та же проблема останется и в сборке без GIL, если цикл событий один.


**Итог для вычислений на чистом Python в CPython с включённой GIL:**

| Модель | Python-код одновременно на разных ядрах | Что происходит со спиннером |
|---|---|---|
| `multiprocessing` | Возможно, в разных процессах | Работает независимо от вычисления |
| `threading` | Нет, потоки чередуются | Получает время между вычислениями |
| `asyncio` в одном потоке | Нет, корутины уступают управление | Не работает, пока CPU-код не вернёт управление |


### Power Napping: `await asyncio.sleep(0)` как «микро-уступка»

- Быстрый хак: переписать `is_prime` как **корутину** и периодически делать `await asyncio.sleep(0)` для отдачи управления.  
- Это «микро-сон» (нулевая задержка), позволяющий event loop **прокачать другие корутины** (в т.ч. спиннер).  
- Минусы:  
  - вычисление станет **медленнее** (на заметную долю);  
  - event loop всё равно будет **замедляться**: CPU-код остаётся внутри того же потока.  
- Это временная мера; правильнее – **делегировать CPU-вычисления** в отдельный процесс/исполнитель.


In [10]:
# spinner_async_nap.py: корутинная версия is_prime с "микро-сном"
import asyncio
import math

async def is_prime_async(n: int) -> bool:
    if n < 2: return False
    if n == 2: return True
    if n % 2 == 0: return False
    root = math.isqrt(n)
    for i in range(3, root + 1, 2):
        if n % i == 0:
            return False
        # каждые ~100_000 итераций "уступаем" event loop
        if i % 100_000 == 1:
            await asyncio.sleep(0)  # нулевая задержка, но отдаём управление
    return True

# Пример интеграции в "slow"
async def slow() -> int:
    n = 5_000_111_000_222_021
    return 42 if await is_prime_async(n) else 0

result = await supervisor()
print(f'Answer: {result}')


spinner task: done=False
Answer: 42


### Самодельный пул процессов: последовательный базовый пример
 
Рассмотрим использование *нескольких процессов* для CPU-интенсивных задач и типичный шаблон работы с **очередями**:  
- одна очередь для *распределения задач* (`jobs`),  
- другая – для *сбора результатов* (`results`).  

В этой части – **базовая версия без параллельности**: программа `sequential.py`, служащая эталоном для измерения ускорения при использовании `multiprocessing`.

**Контекст:**  
- Мы будем проверять **простоту 20 чисел** из диапазона от `2` до `9_999_999_999_999_999` (то есть примерно до \(10^{16} - 1\)).  
- В выборке есть как **простые**, так и **составные** числа с большими и малыми делителями.  
- Время проверки сильно варьируется – это удобно для демонстрации распараллеливания.  


In [ ]:
"""
sequential.py – базовая версия для сравнения с многопроцессной реализацией.
Проверяет набор чисел NUMBERS на простоту последовательно.
"""

from time import perf_counter
from typing import NamedTuple
from primes import is_prime, NUMBERS

class Result(NamedTuple):
    prime: bool
    elapsed: float

def check(n: int) -> Result:
    """Проверка числа n на простоту и измерение времени выполнения."""
    t0 = perf_counter()
    prime = is_prime(n)
    return Result(prime, perf_counter() - t0)

def main() -> None:
    print(f'Checking {len(NUMBERS)} numbers sequentially:')
    t0 = perf_counter()

    for n in NUMBERS:
        prime, elapsed = check(n)
        label = 'P' if prime else ' '
        print(f'{n:16} {label} {elapsed:9.6f}s')

    elapsed = perf_counter() - t0
    print(f'Total time: {elapsed:.2f}s')

if __name__ == '__main__':
    main()

### Ускорение измеряют на одинаковой работе

До сравнения проверьте, что оба варианта получают одинаковые данные и дают
одинаковые результаты. Сравнение успешного расчёта с пропущенными заданиями бессмысленно.

`ускорение = время последовательного варианта / время параллельного варианта`

- Для полного времени запустите `perf_counter()` **до создания процессов**,
  остановите после получения результатов и завершения воркеров.
- В него входят запуск интерпретаторов, передача данных, вычисления и сбор ответов.
  Время одной функции внутри воркера не включает все эти расходы.
- Повторите измерения, сравните медианы. Печать результатов вынесите за границы замера.
- Если пул будет жить долго, его повторное использование измеряют отдельным опытом,
  с явно указанными границами времени.


### Размер задания меняет результат сравнения

Готовый опыт: [`benchmark_procs.py`](../scripts/8_lesson/benchmark_procs.py).
Из корня репозитория:

```text
python scripts/8_lesson/benchmark_procs.py
```

Скрипт проверяет 24 числа последовательно и через два процесса со `spawn`,
сравнивает ответы и берёт медиану трёх полных запусков каждого варианта.

| Набор | Что проверяем |
|---|---|
| 24 раза `104_729` | Короткие задания: оправдает ли вычисление расходы запуска? |
| 24 раза `1_000_000_000_039` | Более долгие задания: окупится ли распределение работы? |

Если ускорение меньше 1, параллельный вариант медленнее. Конкретные числа
зависят от машины, ОС и нагрузки. Большие объёмы передаваемых данных тоже могут
съесть выигрыш: здесь аргументы маленькие, по одному целому числу.


### Последовательная часть ограничивает ускорение

Пусть из 10 секунд две уходят на подготовку, которую нельзя распараллелить,
а восемь – на независимые вычисления. Даже при идеальном распределении:

`T(p) = 2 + 8 / p`, где `p` – число одновременно работающих исполнителей.

Ниже **расчётная модель**, а не замер Python. Она не учитывает запуск, передачу
данных и конкуренцию за ресурсы. При любом числе исполнителей остаются две секунды:
предельное ускорение здесь равно 5. Это ограничение не связано с GIL.


In [27]:
for workers in (1, 2, 4, 8):
    seconds = 2 + 8 / workers
    speedup = 10 / seconds
    print(f"Исполнителей: {workers}; время: {seconds:.1f} с; ускорение: {speedup:.2f}")


Исполнителей: 1; время: 10.0 с; ускорение: 1.00
Исполнителей: 2; время: 6.0 с; ускорение: 1.67
Исполнителей: 4; время: 4.0 с; ускорение: 2.50
Исполнителей: 8; время: 3.0 с; ускорение: 3.33


### Многопроцессная проверка простых чисел: очереди, типы и воркеры 

Цель: распределить CPU-интенсивные проверки по процессам и собирать результаты через очереди.

Ключевые идеи:
- Рабочие процессы не «возвращают» значения напрямую → результаты кладём в **очередь**.
- Две разные очереди: **jobs** (задания) и **results** (ответы и маркеры завершения).

```text
main --put--> jobs --get--> worker --put--> results --get--> report
```
- «Яд» для остановки воркеров – **poison pill** (число `0`).
- `multiprocessing.SimpleQueue` подходит для обмена в этом небольшом примере;
  у неё нет учёта завершённых заданий и таймаута у `get()`.
- Для подсказок типов используем классы из `multiprocessing.queues`.

Структура:
- `PrimeResult(NamedTuple)`: `n`, `prime`, `elapsed`.
- `check(n)`: синхронная проверка с таймингом.
- `worker(jobs, results)`: цикл `get()`, обработка, отправка результата, сигнал завершения.
- `start_jobs(procs, jobs, results)`: запустить воркеров, наполнить `jobs`, раздать `0` как «яд»,
  вернуть процессы.

> Всё это мы пишем руками, чтобы увидеть механику: очереди, воркеры, сентинелы.
> В следующей лекции ту же задачу решим через `ProcessPoolExecutor` – он уместит
> её в несколько строк. Сравнение двух версий и есть смысл упражнения.

Ниже – упрощённый разбор **успешного сценария для 20 чисел**. Он не контролирует
ошибки воркеров. После разбора рассмотрим ограничения; запускаемый `procs.py`
дополнен обработкой ошибок, контролем процессов и сроком ожидания.


In [ ]:
# Учебный пул – часть 1: импорты, типы, функции check/worker/start_jobs
import sys
from time import perf_counter
from typing import NamedTuple
from multiprocessing import Process, SimpleQueue, cpu_count
from multiprocessing import queues  # для аннотаций типов очередей
from primes import is_prime, NUMBERS

class PrimeResult(NamedTuple):
    n: int
    prime: bool
    elapsed: float

# Аннотации типов: используем определения из multiprocessing.queues
JobQueue = queues.SimpleQueue[int]
ResultQueue = queues.SimpleQueue[PrimeResult]

def check(n: int) -> PrimeResult:
    t0 = perf_counter()
    res = is_prime(n)
    return PrimeResult(n, res, perf_counter() - t0)

def worker(jobs: JobQueue, results: ResultQueue) -> None:
    """
    Берёт задания из jobs, кладёт PrimeResult в results.
    Останавливается, получив 'яд' (0) и отправляет маркер завершения.
    """
    while n := jobs.get():                  # получить задание
        results.put(check(n))               # ответить в другую очередь
    # Сообщаем главному процессу о завершении этого воркера
    results.put(PrimeResult(0, False, 0.0))

def start_jobs(procs: int, jobs: JobQueue, results: ResultQueue) -> list[Process]:
    """
    Запускает воркеров, затем отправляет задания и маркеры остановки.
    Возвращает процессы, чтобы main дождался их завершения.
    """
    processes = []
    for _ in range(procs):
        proc = Process(target=worker, args=(jobs, results))
        proc.start()
        processes.append(proc)
    for n in NUMBERS:
        jobs.put(n)                         # сначала все задания
    for _ in range(procs):
        jobs.put(0)                         # затем по остановке на воркер
    return processes


### Цикл воркера, сентинелы и «ядовитые пилюли»

Ключевой паттерн:
- **Бесконечный цикл**: воркер берёт элементы из очереди `jobs` и обрабатывает их.
- **Сентинел (poison pill)**: специальное значение в `jobs` для **корректной остановки** воркера.
- Выбор сентинела:
  - `None` – популярный вариант, но не годится, если может встретиться в данных.
  - `object()` – уникален в своём процессе, но после передачи в другой процесс становится
    **новым объектом**: проверка `is` не сработает.
  - **`Ellipsis` (`...`)** – хорош для межпроцессной передачи (сохраняет тождество при сериализации).
  - В примере – число **`0`**: простая «пилюля», т.к. в данных `0` не встречается.
- Каждому воркеру – **свой** сентинел (один на процесс).
- У `queue.Queue` и `asyncio.Queue` в Python 3.13+ вместо сентинелов можно
  использовать `shutdown()`; практический шаблон показан ниже.
  У очередей `multiprocessing` этого метода нет.
- Отчёт о завершении: воркер может положить **маркер завершения** в `results` (напр. `PrimeResult(0, False, 0.0)`).


In [ ]:
# Учебный пул – часть 2: main и report 
import sys
from time import perf_counter
from multiprocessing import SimpleQueue, cpu_count

# from multiprocessing import queues
# from typing import NamedTuple
# from primes import is_prime, NUMBERS
# from <this_module> import PrimeResult, JobQueue, ResultQueue, start_jobs

def report(procs: int, results: "ResultQueue") -> int:
    """
    Читает результаты из очереди 'results' до тех пор, пока не завершатся все 'procs' воркеров.
    Возвращает количество проверенных чисел.
    """
    checked = 0
    procs_done = 0
    while procs_done < procs:
        n, prime, elapsed = results.get()           # блокирующее чтение результата
        if n == 0:                                   # маркер завершения воркера
            procs_done += 1
        else:
            checked += 1
            label = 'P' if prime else ' '
            print(f'{n:16} {label} {elapsed:9.6f}s')
    return checked

def main() -> None:
    # определяем число процессов из аргумента командной строки или по числу CPU
    if len(sys.argv) < 2:
        procs = cpu_count()
    else:
        procs = int(sys.argv[1])

    print(f'Checking {len(NUMBERS)} numbers with {procs} processes:')

    t0 = perf_counter()
    jobs: "JobQueue" = SimpleQueue()
    results: "ResultQueue" = SimpleQueue()

    processes = start_jobs(procs, jobs, results)     # запустить воркеров и раздать задания
    checked = report(procs, results)                 # собрать и вывести результаты
    for proc in processes:
        proc.join()                                  # воркер получил «яд», ждём выхода процесса

    elapsed = perf_counter() - t0
    print(f'{checked} checks in {elapsed:.2f}s')

if __name__ == '__main__':
    main()


### Если воркер пула не прислал результат

В учебном `worker` ошибка `check(n)` прервёт функцию до отправки маркера завершения.
Тогда `report()` может бесконечно ждать следующий ответ. Перенос маркера в `finally`
сам по себе не сообщит, какое задание потеряно.

В запускаемом [`procs.py`](../scripts/8_lesson/procs.py) предусмотрены три случая:

| Случай | Реакция |
|---|---|
| Проверка числа подняла обычное исключение | Воркер отправляет `Result` с полем `error` и берёт следующее задание |
| Процесс завершился без ответа | Сборщик проверяет `exitcode` и число полученных сообщений |
| Процесс жив, но не отвечает | Истекает общий срок ожидания, заданный `timeout` |

Файловая версия ждёт по одному ответу на задание, затем завершения процессов.
При аварии оставшиеся процессы останавливаются, очереди больше не используются.
Сообщение об ошибке и повтор задания – разные решения; автоматического повтора здесь нет.


### Пустая очередь и завершённая обработка – разные состояния

Рассмотрим `queue.Queue`, которая умеет учитывать незавершённые задания:

| Действие | Элементов в очереди | Неподтверждённых заданий |
|---|---:|---:|
| `put("отчёт")` | 1 | 1 |
| Воркер сделал `get()` | 0 | 1 |
| Закончил обработку и вызвал `task_done()` | 0 | 0 |

`Queue.join()` ждёт нулевого числа неподтверждённых заданий.
`Thread.join()` ждёт завершения потока. Это два разных условия.

`empty()` не проверяет окончание обработки и не гарантирует, что следующий `get()`
не заблокируется: между вызовами очередь может измениться. У `SimpleQueue`
нет `task_done()` и `join()`; для процессов аналог с учётом заданий – `JoinableQueue`.


### Одно подтверждение на каждое полученное задание

Вызовите `task_done()` в `finally` после успешного `get()` – в том числе для
ошибочного задания и полученного маркера остановки. Здесь обработка ошибки
завершает попытку; успех и ошибка записываются отдельно.

Пропущенное подтверждение оставит `Queue.join()` ждать. Лишнее подтверждение
может преждевременно обнулить счётчик, а при превышении числа `put()` вызовет `ValueError`.


In [28]:
from queue import Queue
from threading import Thread

jobs = Queue()
outcomes = []

def consume():
    while True:
        item = jobs.get(timeout=2)
        try:
            if item is None:
                return
            try:
                outcomes.append(len(item))
            except TypeError:
                outcomes.append("ошибка типа")
        finally:
            jobs.task_done()

consumer = Thread(target=consume)
consumer.start()
for item in ("abc", 42, "python", None):
    jobs.put(item)
jobs.join()
consumer.join(timeout=2)
if consumer.is_alive():
    raise TimeoutError("Поток не завершился")
print(outcomes)


[3, 'ошибка типа', 6]


### `Queue.shutdown()`: производитель закрывает вход

Производитель вызывает `shutdown()` **после последнего `put()`**. Новые задания
больше не принимаются, а уже добавленные обрабатываются. Когда очередь опустеет,
`get()` поднимет `ShutDown`, и потребитель завершит цикл без сентинела.

```python
from queue import Queue, ShutDown

def consume():
    try:
        while True:
            item = jobs.get()
            try:
                handle(item)
            finally:
                jobs.task_done()
    except ShutDown:
        pass

for item in items:
    jobs.put(item)
jobs.shutdown()
jobs.join()
```


### `asyncio.Queue.shutdown()`: тот же жизненный цикл

В асинхронном варианте производитель тоже закрывает вход после последнего
`await jobs.put(...)`. Сам `shutdown()` не нужно ожидать через `await`.

```python
async def consume():
    try:
        while True:
            item = await jobs.get()
            try:
                await handle(item)
            finally:
                jobs.task_done()
    except asyncio.QueueShutDown:
        pass

for item in items:
    await jobs.put(item)
jobs.shutdown()
await jobs.join()
```


### Ограниченная очередь сдерживает производителя

Если задания поступают быстрее обработки, очередь растёт.
`Queue(maxsize=2)` ограничивает число **ожидающих элементов**: следующий `put()`
ждёт свободного места либо поднимает `Full` по таймауту.

Это не ограничение размера элементов в байтах и не число работающих потоков.
Место освобождается при `get()`, а не при `task_done()`.


In [29]:
from queue import Queue, Full

pending = Queue(maxsize=2)
pending.put("A")
pending.put("B")
try:
    pending.put("C", timeout=0.01)
except Full:
    print("Мест нет: третье задание пока не принято")
print("Получено:", pending.get())
pending.put("C", timeout=0.01)
print("После get третье задание принято")


Мест нет: третье задание пока не принято
Получено: A
После get третье задание принято


### Сначала запустить потребителей, затем наполнять очередь

Ошибочный порядок при `maxsize=1`: положить первое задание, пытаться положить
второе, а запуск потребителя оставить на потом. Второй `put()` ждёт места,
но освобождать его некому. Один большой предварительный список заданий этого не исправляет.

Используем уже определённый `consume`: запускаем его до отправки данных.
Остановка знакомым маркером и подтверждения обработки сохраняются.


In [30]:
jobs = Queue(maxsize=1)
outcomes = []
consumer = Thread(target=consume)
consumer.start()
for item in ("one", "two", None):
    jobs.put(item, timeout=2)
jobs.join()
consumer.join(timeout=2)
if consumer.is_alive():
    raise TimeoutError("Поток не завершился")
print("Результаты:", outcomes)


Результаты: [3, 3]


## Частые ошибки

- **Ждать ускорения чистого Python от потоков с включённой GIL.**
  Сопоставьте задачу с разобранными ограничениями модели.
- **Считать удачный запуск доказательством отсутствия гонки.**
  Проверьте, что один Lock охватывает проверку и изменение общих данных.
- **Принять возврат из `join(timeout)` за завершение.** Проверьте `is_alive()`.
- **Запускать процессы на верхнем уровне модуля.** Определите рабочую функцию
  на уровне модуля, а `Process(...)`, `start()` и основной сценарий поместите
  под `if __name__ == '__main__':`.
- **Блокирующий вызов в `asyncio`** останавливает весь событийный цикл.
- **Считать `cancel()` остановкой задачи.** Это запрос: без последующего `await`
  очистка может ещё выполняться. Подавленный `CancelledError` заставит задачу
  выглядеть успешно завершённой и может нарушить работу `TaskGroup` и таймаутов.

- **Считать окончание потока успехом задания.** Проверяйте явный ответ воркера.
- **Приравнивать пустую очередь к завершённой работе.** Используйте разобранный
  учёт заданий; не забывайте подтверждать и ошибочные попытки.
- **Измерять только сбор ответов.** Для полного времени включайте запуск и завершение.


## Мини-упражнения


### Потокобезопасный счётчик

Реализуйте счётчик с `threading.Lock`: 8 потоков по 10 000 инкрементов должны дать ровно
**80 000**.

Без блокировки результат тоже может оказаться правильным: это зависит от реализации,
версии Python и порядка выполнения. Такой запуск не доказывает безопасность.
Воспроизводимый пример ошибки уже разобран на продаже последнего билета.


In [ ]:
import threading

def run(use_lock):
    counter = 0
    lock = threading.Lock()
    def work():
        nonlocal counter
        for _ in range(10_000):
            if use_lock:
                with lock:
                    counter += 1
            else:
                counter += 1
    threads = [threading.Thread(target=work) for _ in range(8)]
    for t in threads: t.start()
    for t in threads: t.join()
    return counter

print("с блокировкой:", run(use_lock=True))

с блокировкой: 80000


### Какую модель выбрать

Для каждой задачи выберите `threading`, `multiprocessing` или `asyncio` и объясните выбор.

1. Скачать 50 файлов обычной блокирующей HTTP-библиотекой: загрузка почти всё время ждёт сеть.
2. Проверить на простоту 20 больших чисел функцией на чистом Python.
3. Сжать модулем `zlib` восемь файлов по 100 МБ, уже прочитанных в память.
4. Держать на сервере чата 10 000 подключений, большинство из которых подолгу молчит.


### Ответы: какую модель выбрать

| Задача | Модель | Почему |
|---|---|---|
| 1. Загрузка файлов | `threading` | Ожидание сети освобождает GIL, потоки перекрывают ожидания. Процессы добавят расходы на запуск без выигрыша |
| 2. Простые числа | `multiprocessing` | С включённой GIL потоки не исполняют Python-код одновременно, а у каждого процесса своя GIL |
| 3. Сжатие `zlib` | `threading` | CPython освобождает GIL вокруг вызова нативной библиотеки zlib: потоки работают на разных ядрах и не передают 100 МБ между процессами |
| 4. Чат | `asyncio` | Корутина дешевле потока, ожидание отдаёт управление циклу событий. Все вызовы должны быть асинхронными: блокирующий остановит все подключения |

Процессы тоже могут сжимать параллельно, но здесь восемь буферов уже находятся
в памяти родителя: их сериализация, копирование и отдельная память добавляют расходы.
Если каждый процесс сам читает свой файл, результат может отличаться – нужно измерять.


## Что почитать

- [`threading` в Python 3.13](https://docs.python.org/3.13/library/threading.html) –
  GIL, `join`, блокировки и события.
- [`multiprocessing`: требования spawn](https://docs.python.org/3.13/library/multiprocessing.html#the-spawn-and-forkserver-start-methods) –
  импорт главного модуля, функции и передаваемые аргументы.
- [`sys.setswitchinterval`](https://docs.python.org/3.13/library/sys.html#sys.setswitchinterval) –
  почему интервал переключения не задаёт точное расписание.
- [`queue.Queue`](https://docs.python.org/3.13/library/queue.html) – размер очереди,
  подтверждения обработки и ожидание завершения.
- [asyncio: отмена задач](https://docs.python.org/3.13/library/asyncio-task.html#task-cancellation) –
  почему `CancelledError` не подавляют и как дождаться отменённой задачи.
- [Python без GIL](https://docs.python.org/3.14/howto/free-threading-python.html) –
  как установить и проверить free-threaded сборку, какие у неё ограничения.
- Л. Рамальо, «Python. К вершинам мастерства», 2-е изд. – часть про параллелизм и GIL;
  книга написана под Python 3.10.
